# Backpropagation with a Tiny Network

This notebook works through one training example in a tiny neural network. You will calculate a prediction, measure its error, and use the chain rule to find how each parameter affected that error.

The example uses plain Python and one input value so the arithmetic is visible. Real models use automatic differentiation to perform the same kind of chain-rule calculation efficiently across large computational graphs.

## Learning goals

By the end, you should be able to:

- Distinguish a forward pass from a backward pass.
- Follow the chain rule from a loss back to earlier parameters.
- Explain how gradients are used by an optimizer.
- Check a manually calculated gradient with a numerical approximation.

We will use a one-hidden-unit network. It is intentionally small; the same reasoning extends to larger differentiable models.

## 1. Define a tiny network

The network has one input $x$, a hidden unit with a `tanh` activation, and one output. Its parameters are $w_1$, $b_1$, $w_2$, and $b_2$:

$$z_1 = w_1x+b_1, \qquad h = \tanh(z_1)$$
$$\hat{y} = w_2h+b_2$$

We use squared error $L = (\hat{y}-y)^2$ for a target $y$. The code below stores a small example's inputs and parameters. Run it and inspect the values.

In [ ]:
from math import tanh

x = 1.5
target = 0.4
w1, b1 = 0.8, -0.1
w2, b2 = -0.7, 0.2

x, target, w1, b1, w2, b2

## 2. Run the forward pass

The forward pass follows the network from input to output. First calculate the hidden unit's pre-activation $z_1$, then its activated value $h$, then the prediction $\hat{y}$ and loss $L$.

Run the cell and check that the loss is a single number. It tells us how far this prediction is from the chosen target under squared error.

In [ ]:
z1 = w1 * x + b1
h = tanh(z1)
prediction = w2 * h + b2
loss = (prediction - target) ** 2

z1, h, prediction, loss

## 3. Start at the loss

Backpropagation applies the chain rule from the end of the computation toward its beginning. For squared error, the derivative of the loss with respect to the prediction is:

$$\frac{\partial L}{\partial \hat{y}} = 2(\hat{y}-y)$$

The output is $\hat{y}=w_2h+b_2$, so its gradients with respect to $w_2$, $b_2$, and $h$ follow directly. Calculate them below. Each variable name describes the derivative being stored.

In [ ]:
d_loss_d_prediction = 2 * (prediction - target)
d_prediction_dw2 = h
d_prediction_db2 = 1.0
d_prediction_dh = w2

d_loss_dw2 = d_loss_d_prediction * d_prediction_dw2
d_loss_db2 = d_loss_d_prediction * d_prediction_db2
d_loss_dh = d_loss_d_prediction * d_prediction_dh

d_loss_d_prediction, d_loss_dw2, d_loss_db2, d_loss_dh

## 4. Continue through the hidden unit

The hidden unit is $h=\tanh(z_1)$. Its derivative is $1-h^2$. The pre-activation is $z_1=w_1x+b_1$, so the chain rule gives the gradients for the first-layer parameters:

$$\frac{\partial L}{\partial w_1} = \frac{\partial L}{\partial h}(1-h^2)x$$
$$\frac{\partial L}{\partial b_1} = \frac{\partial L}{\partial h}(1-h^2)$$

Run the calculation. These gradients tell us the direction and sensitivity of the loss with respect to each parameter at the current parameter values.

In [ ]:
d_hidden_d_z1 = 1 - h**2
d_z1_dw1 = x
d_z1_db1 = 1.0

d_loss_dw1 = d_loss_dh * d_hidden_d_z1 * d_z1_dw1
d_loss_db1 = d_loss_dh * d_hidden_d_z1 * d_z1_db1

d_loss_dw1, d_loss_db1

## 5. Collect the gradients

A gradient is the derivative of the loss with respect to a parameter. The four gradients are ready for an optimizer. A simple gradient-descent update uses learning rate $\eta$:

$$w \leftarrow w - \eta\frac{\partial L}{\partial w}$$

The learning rate controls the step size; backpropagation supplies the gradients but does not choose that step size. Inspect the gradients together before making an update.

In [ ]:
gradients = {
    "w1": d_loss_dw1,
    "b1": d_loss_db1,
    "w2": d_loss_dw2,
    "b2": d_loss_db2,
}

gradients

## 6. Check one gradient numerically

A finite-difference estimate changes one parameter by a tiny amount and observes the change in loss:

$$\frac{\partial L}{\partial w_1} \approx \frac{L(w_1+\epsilon)-L(w_1-\epsilon)}{2\epsilon}$$

This is a useful check for a small example, although it is slower and less numerically reliable than analytic or automatic differentiation for large models. Compare this estimate with the chain-rule result.

In [ ]:
def loss_for_w1(candidate_w1):
    candidate_h = tanh(candidate_w1 * x + b1)
    candidate_prediction = w2 * candidate_h + b2
    return (candidate_prediction - target) ** 2

epsilon = 1e-5
numerical_gradient_w1 = (
    loss_for_w1(w1 + epsilon) - loss_for_w1(w1 - epsilon)
) / (2 * epsilon)
(d_loss_dw1, numerical_gradient_w1)

The two values should be close, with a small difference due to finite precision and the approximation. If they differ substantially, revisit the forward-pass equations and the chain-rule factors.

## 7. Make one gradient-descent update

Try a small learning rate and update each parameter by subtracting the learning rate times its gradient. Then run the forward pass again using the updated parameters and compare the loss.

For this one example, a small update may reduce the loss, but that is not guaranteed for every learning rate. In real training, gradients are computed over batches, and optimizers often include additional state or adaptive rules.

In [ ]:
learning_rate = 0.1

w1 -= learning_rate * d_loss_dw1
b1 -= learning_rate * d_loss_db1
w2 -= learning_rate * d_loss_dw2
b2 -= learning_rate * d_loss_db2

z1 = w1 * x + b1
h = tanh(z1)
prediction = w2 * h + b2
updated_loss = (prediction - target) ** 2

loss, updated_loss

## What to remember

- The forward pass computes predictions and a loss.
- Backpropagation applies the chain rule to compute loss gradients through the computation graph.
- An optimizer uses those gradients to update parameters.
- Numerical gradient checks can help validate a small hand-computed example.
- This scalar example illustrates the idea; practical neural networks use automatic differentiation and many examples per update.

Further experiment: change the input or target, recompute the forward pass and gradients, and observe how the update changes. Make sure you recompute gradients after changing parameters.